In [1]:
"""
exp08 Step 01: The VIX Gate (PROTOCOL.md §4-§8)

Four pre-registered gates decide which VIX experiments may run (primary timing _prev only):
    G1  UNIVARIATE: exp02's daily signal check on the six VIX features (training 2009-08-12 -> 2015-03-18, validation =
        the 44 replay periods pooled); S1 = number of signal (feature, bin) pairs; PASS if S1 > max of 19 shifted nulls.
    G2  INCREMENTAL: exp02's logistic regression per fold (expanding training), BASE (16 price features) vs AUGMENTED
        (+ 6 VIX features) on the same rows; dAUC of the pooled out-of-sample predictions; PASS if dAUC > max of 19 nulls.
    G3  VOLATILITY FORECAST: D = Spearman(vix_level_prev, forward 20-session volatility) - Spearman(daily_volatility, same)
        on the validation decisions; PASS if the 95% lower bound (6-month circular blocks) > 0.
    G4  PLAUSIBILITY OF UNLEVERED SCALING: mean forward 20-session net return of the top VIX quintile (training 80th
        percentile); PASS only if below 0 on training AND on validation.
    SECONDARY (reported, never a gate): G1 and G2 with the _intraday features.
Gate rules: exp09 if G1 or G2; exp11 if G2; exp10 if G3 and G4; none -> the VIX line stops.

Data: SPY bars and every VIX load cut at 2026-05-13 (the 20-session labels and forward volatilities of the last
validation decision end on that session, the last before the untouched window). Nothing dated 2026-05-14 or later is
used. Trial log: 6 entries (stage "signal_check"; budget 10). Outputs: store04_experiments/exp08_vix_signal_check/step01_gate_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import time
import numpy as np
import pandas as pd
# IMPORT THE SHARED CONFIGURATION, THE VIX CONSTANTS, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths, vix_config
from experiments.exp08_vix_signal_check import config as exp_config
from experiments.exp02_stop_reentry import config as exp02_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT THE VIX DATA LAYER
from so.features.vix_features import get_vix_daily_feature_dict, get_vix_feature_col_list
# IMPORT THE SCHEDULE (exp04 step 02's) AND THE REPLAY PERIODS
from experiments.exp04_trend_exit.rules import get_schedule_tuple
from so.core.continuous_replay import get_replay_period_pdf
# IMPORT THE GATES
from experiments.exp08_vix_signal_check.gates import (check_schedule_dict, get_analysis_pdf, get_null_dict, run_g1_dict, run_g2_dict, get_auc_interval_dict,
                                                      run_g3_dict, run_g4_dict, get_gate_decision_dict, get_descriptive_return_pdf, get_term_episode_pdf)
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning
# DISPLAY ALL COLUMNS OF THE TABLES
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)

In [3]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# DEFINE WHETHER THE REAL SCHEDULE IS ASSERTED (ALWAYS IN A REAL RUN; THE SYNTHETIC SMOKE RUN HAS A SHORTER SCHEDULE)
CHECK_REAL_SCHEDULE_BOOL = __name__ != "__smoke__"
# DEFINE THE FEATURE COLUMNS OF BOTH TIMINGS
PRIMARY_COL_STR_LIST = get_vix_feature_col_list(exp_config.PRIMARY_VARIANT_STR)
SECONDARY_COL_STR_LIST = get_vix_feature_col_list(exp_config.SECONDARY_VARIANT_STR)
# ASSERT THAT THE MODEL AND SIGNAL CHECK CONSTANTS ARE exp02's
assert exp_config.LOGISTIC_C == exp02_config.LOGISTIC_C and exp_config.SIGNAL_CHECK_BIN_COUNT == exp02_config.DAILY_SIGNAL_CHECK_BIN_COUNT
assert exp_config.SIGNAL_CHECK_MIN_BIN_COUNT == exp02_config.DAILY_SIGNAL_CHECK_MIN_BIN_COUNT and exp_config.SIGNAL_CHECK_MIN_MONTH_COUNT == exp02_config.DAILY_SIGNAL_CHECK_MIN_MONTH_COUNT
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tPrimary:\t{PRIMARY_COL_STR_LIST}\nSecondary:\t{SECONDARY_COL_STR_LIST}")
print(f"Null runs:\t{exp_config.NULL_RUN_COUNT}\tshift share {exp_config.NULL_SHIFT_SHARE_RANGE}\tseed {exp_config.NULL_SEED}\tData cutoff:\t{exp_config.DATA_CUTOFF_DATE_STR}")

Run Mode:	validation_only	Primary:	['vix_level_prev', 'vix_pct250_prev', 'vix_chg5_prev', 'vix_fade_prev', 'vix_term_prev', 'vrp_prev']
Secondary:	['vix_level_intraday', 'vix_pct250_intraday', 'vix_chg5_intraday', 'vix_fade_intraday', 'vix_term_intraday', 'vrp_intraday']
Null runs:	19	shift share (0.25, 0.75)	seed 20261009	Data cutoff:	2026-05-13


In [4]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_gate_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "signal_check")

Experiment:	exp08_vix_signal_check	Protocol:	1.0	Config Hash:	43e86ccb06
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp08_vix_signal_check/step01_gate_data/
ℹ️ No earlier 'signal_check' entry of exp08_vix_signal_check in the trial log (config hash 43e86ccb06).


In [5]:
# CALL FUNCTION TO READ THE MINUTE BARS AND BUILD THE SCHEDULE FROM THE FULL SESSION CALENDAR (exp04 step 02: SAME 44 QUARTERS)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
fold_pdf, _ = get_schedule_tuple(sorted(complete_ohlcv_pdf["date"].unique()))
schedule_dict = check_schedule_dict(fold_pdf) if CHECK_REAL_SCHEDULE_BOOL else {"fold_count": len(fold_pdf)}
print(f"Schedule:\t{schedule_dict}")
# CUT THE BARS AT THE DATA CUTOFF BEFORE ANY COMPUTATION (NO LATER PRICE IS USED)
cutoff_date = pd.Timestamp(exp_config.DATA_CUTOFF_DATE_STR).date()
assert cutoff_date < pd.Timestamp(exp_config.UNTOUCHED_START_DATE_STR).date()
cut_ohlcv_pdf = complete_ohlcv_pdf[complete_ohlcv_pdf["date"] <= cutoff_date].reset_index(drop=True)
del complete_ohlcv_pdf
# CALL FUNCTION TO BUILD THE DAILY TABLE AND THE REPLAY PERIODS OF THE CUT DATA
ohlcv_array_dict = get_ohlcv_array_dict(cut_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
period_pdf = get_replay_period_pdf(fold_pdf, daily_pdf["date"].tolist())
# CALL FUNCTION TO LOAD THE VIX AND VIX3M BARS WITH THE SAME CUTOFF AND BUILD THE FEATURES OF BOTH TIMINGS
vix_dict = get_vix_daily_feature_dict(daily_pdf, exp_config.DATA_CUTOFF_DATE_STR, [exp_config.PRIMARY_VARIANT_STR, exp_config.SECONDARY_VARIANT_STR])
vix_feature_pdf = vix_dict["feature_pdf"]
assert (vix_feature_pdf["session_idx"].to_numpy() == daily_pdf["session_idx"].to_numpy()).all()
# DISPLAY THE LAST DATES LOADED
print(f"Last SPY date loaded:\t{cut_ohlcv_pdf['date'].max()}\t({len(daily_pdf):,} sessions from {daily_pdf['date'].iloc[0]})")
for key_str, last_date in vix_dict["last_date_dict"].items():
    print(f"Last date loaded:\t{key_str}\t{last_date}")
assert cut_ohlcv_pdf["date"].max() <= cutoff_date and all(d <= cutoff_date for d in vix_dict["last_date_dict"].values())
print(f"Replay periods:\t{len(period_pdf)}\t{period_pdf['period_start'].iloc[0]} -> {period_pdf['period_end'].iloc[-1]}\t({int(period_pdf['session_count'].sum()):,} sessions)")

Schedule:	{'fold_count': 44, 'first_train_end': '2015-03-18', 'first_valid_start': '2015-04-17', 'last_valid_end': '2026-04-15', 'latest_test_start': '2026-05-14'}


Last SPY date loaded:	2026-05-13	(5,373 sessions from 2005-01-03)
Last date loaded:	vix_daily	2026-05-13
Last date loaded:	vix3m_daily	2026-05-13
Last date loaded:	vix_1min	2026-05-13
Last date loaded:	vix3m_1min	2026-05-13
Replay periods:	44	2015-04-17 -> 2026-04-15	(2,765 sessions)


In [6]:
"""
Analysis rows (one table per timing): sessions from 2009-08-12 to the last validation session with the label and all 22
features known. Training = rows up to the first fold's train_end; validation = rows of the 44 replay periods.
"""
# BUILD THE ANALYSIS ROWS OF BOTH TIMINGS
analysis_pdf_dict = {variant_str: get_analysis_pdf(daily_pdf, vix_feature_pdf, fold_pdf, get_vix_feature_col_list(variant_str))
                     for variant_str in [exp_config.PRIMARY_VARIANT_STR, exp_config.SECONDARY_VARIANT_STR]}
# DISPLAY AND CHECK THE ROWS
for variant_str, analysis_pdf in analysis_pdf_dict.items():
    train_pdf, valid_pdf = analysis_pdf[analysis_pdf["is_train"]], analysis_pdf[analysis_pdf["valid_fold_id"] >= 0]
    print(f"{variant_str}:\t{len(analysis_pdf):,} rows {analysis_pdf['date'].iloc[0]} -> {analysis_pdf['date'].iloc[-1]}\ttraining {len(train_pdf):,} ({train_pdf['date'].iloc[0]} -> {train_pdf['date'].iloc[-1]}, "
          f"base rate {train_pdf['y_fwd_positive'].mean():.3f})\tvalidation {len(valid_pdf):,} ({valid_pdf['date'].iloc[0]} -> {valid_pdf['date'].iloc[-1]}, {valid_pdf['valid_fold_id'].nunique()} periods, "
          f"base rate {valid_pdf['y_fwd_positive'].mean():.3f})\tlargest label end {analysis_pdf['label_end_date'].max()}")
    # ASSERT THE WINDOWS (THE LARGEST SESSION USED BY A LABEL IS ON OR BEFORE THE CUTOFF)
    assert analysis_pdf["label_end_date"].max() <= cutoff_date
    if CHECK_REAL_SCHEDULE_BOOL:
        assert str(train_pdf["date"].iloc[-1]) <= exp_config.EXPECTED_FIRST_TRAIN_END_STR and str(valid_pdf["date"].iloc[-1]) == exp_config.EXPECTED_LAST_VALID_END_STR
        assert str(valid_pdf["date"].iloc[0]) == exp_config.EXPECTED_FIRST_VALID_START_STR and valid_pdf["valid_fold_id"].nunique() == exp_config.EXPECTED_FOLD_COUNT
primary_pdf, secondary_pdf = analysis_pdf_dict[exp_config.PRIMARY_VARIANT_STR], analysis_pdf_dict[exp_config.SECONDARY_VARIANT_STR]

prev:	4,193 rows 2009-08-13 -> 2026-04-15	training 1,408 (2009-08-13 -> 2015-03-18, base rate 0.686)	validation 2,765 (2015-04-17 -> 2026-04-15, 44 periods, base rate 0.671)	largest label end 2026-05-13


intraday:	4,194 rows 2009-08-12 -> 2026-04-15	training 1,409 (2009-08-12 -> 2015-03-18, base rate 0.686)	validation 2,765 (2015-04-17 -> 2026-04-15, 44 periods, base rate 0.671)	largest label end 2026-05-13


In [7]:
"""
G1 UNIVARIATE (primary _prev): exp02's daily signal check on the six VIX features, then 19 session-shifted nulls.
"""
# RUN THE REAL CHECK
start_time = time.time()
g1_dict = run_g1_dict(primary_pdf, PRIMARY_COL_STR_LIST)
print(f"G1 real:\tS1 = {g1_dict['s1']} signal pairs of {g1_dict['test_count']} tested bins\t{g1_dict['signal_pair_list']}")
# RUN THE NULL (BIN EDGES RE-LEARNED IN EVERY RUN)
g1_null_dict = get_null_dict(primary_pdf, PRIMARY_COL_STR_LIST, lambda p: run_g1_dict(p, PRIMARY_COL_STR_LIST)["s1"], g1_dict["s1"])
g1_passed = bool(g1_null_dict["passed"] and g1_dict["s1"] > 0)
print(f"G1 null:\tvalues {g1_null_dict['null_value_list']}\tmax {g1_null_dict['null_max']:.0f}\tp = {g1_null_dict['p_value']:.2f}\tPASS:\t{g1_passed}\t({time.time() - start_time:.0f} s)")
# DISPLAY THE TESTED BINS
g1_dict["signal_check_pdf"][["feature", "bin", "tested", "train_row_count", "train_month_count", "train_rate", "train_rate_ci_low", "train_rate_ci_high", "train_base_rate",
                             "valid_row_count", "valid_month_count", "valid_rate", "valid_rate_ci_low", "valid_rate_ci_high", "valid_base_rate", "signal"]].round(4)

G1 real:	S1 = 0 signal pairs of 30 tested bins	[]


	null run 1/19: shift 2040 rows, value 0.0000


	null run 2/19: shift 2877 rows, value 0.0000


	null run 3/19: shift 1940 rows, value 0.0000


	null run 4/19: shift 2328 rows, value 0.0000


...

	null run 16/19: shift 2638 rows, value 0.0000


	null run 17/19: shift 1384 rows, value 0.0000


	null run 18/19: shift 2698 rows, value 0.0000


	null run 19/19: shift 3118 rows, value 0.0000
G1 null:	values [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]	max 0	p = 1.00	PASS:	False	(5 s)


,feature,bin,tested,train_row_count,train_month_count,train_rate,train_rate_ci_low,train_rate_ci_high,train_base_rate,valid_row_count,valid_month_count,valid_rate,valid_rate_ci_low,valid_rate_ci_high,valid_base_rate,signal
0,vix_level_prev,"(22.936, inf]",True,282,27,0.7199,0.6158,0.8512,0.6861,538,56,0.7398,0.6534,0.8163,0.6712,False
1,vix_level_prev,"(18.562, 22.936]",True,281,39,0.7687,0.6440,0.8747,0.6861,530,74,0.5981,0.4786,0.7069,0.6712,False
2,vix_level_prev,"(16.258, 18.562]",True,282,42,0.7305,0.5943,0.8566,0.6861,412,71,0.6845,0.5909,0.7773,0.6712,False
3,vix_level_prev,"(13.874, 16.258]",True,281,41,0.5801,0.4315,0.7213,0.6861,521,75,0.6948,0.6024,0.7889,0.6712,False
4,vix_level_prev,"(-inf, 13.874]",True,282,24,0.6312,0.4773,0.7793,0.6861,764,61,0.6505,0.5592,0.7479,0.6712,False
5,vix_pct250_prev,"(0.076, 0.22]",True,276,47,0.6594,0.5194,0.7877,0.6861,414,71,0.6691,0.5701,0.7557,0.6712,False
6,vix_pct250_prev,"(-inf, 0.076]",True,289,36,0.6263,0.4595,0.7675,0.6861,418,57,0.7297,0.6273,0.8195,0.6712,False
7,vix_pct250_prev,"(0.22, 0.445]",True,280,52,0.7286,0.6109,0.8208,0.6861,503,84,0.6223,0.5379,0.7141,0.6712,False
8,vix_pct250_prev,"(0.445, 0.728]",True,285,47,0.6947,0.5728,0.8063,0.6861,702,100,0.6709,0.5801,0.7588,0.6712,False
9,vix_pct250_prev,"(0.728, inf]",True,278,32,0.7230,0.6219,0.8498,0.6861,728,78,0.6731,0.5895,0.7598,0.6712,False


In [8]:
"""
G2 INCREMENTAL (primary _prev): BASE (16 price features) vs AUGMENTED (+ 6 VIX features), exp02's logistic regression
refitted for every fold on 2009-08-12 -> train_end(f); pooled out-of-sample AUC over the 44 periods; 19 shifted nulls
(AUGMENTED refitted on every fold in every run, BASE unchanged).
"""
# RUN THE REAL CHECK
start_time = time.time()
g2_dict = run_g2_dict(primary_pdf, fold_pdf, PRIMARY_COL_STR_LIST)
g2_interval_dict = get_auc_interval_dict(g2_dict["base_prediction_pdf"], g2_dict["augmented_prediction_pdf"])
print(f"G2 real:\tAUC BASE {g2_dict['auc_base']:.4f} [{g2_interval_dict['auc_base_ci_low']:.4f}, {g2_interval_dict['auc_base_ci_high']:.4f}]\t"
      f"AUC AUGMENTED {g2_dict['auc_augmented']:.4f} [{g2_interval_dict['auc_augmented_ci_low']:.4f}, {g2_interval_dict['auc_augmented_ci_high']:.4f}]\t"
      f"dAUC {g2_dict['d_auc']:.4f} [{g2_interval_dict['d_auc_ci_low']:.4f}, {g2_interval_dict['d_auc_ci_high']:.4f}] (1-month blocks)\tmean per-period dAUC {g2_dict['mean_quarter_d_auc']:.4f}")
print(f"Prediction rows:\t{len(g2_dict['base_prediction_pdf']):,}\ttraining rows of the first / last fold:\t{g2_dict['base_prediction_pdf']['train_row_count'].iloc[0]:,} / {g2_dict['base_prediction_pdf']['train_row_count'].iloc[-1]:,}")
# RUN THE NULL
g2_null_dict = get_null_dict(primary_pdf, PRIMARY_COL_STR_LIST, lambda p: run_g2_dict(p, fold_pdf, PRIMARY_COL_STR_LIST, base_prediction_pdf_in=g2_dict["base_prediction_pdf"])["d_auc"], g2_dict["d_auc"])
g2_passed = bool(g2_null_dict["passed"])
print(f"G2 null:\tmax {g2_null_dict['null_max']:.4f}\tp = {g2_null_dict['p_value']:.2f}\tPASS:\t{g2_passed}\t({time.time() - start_time:.0f} s)")
# DISPLAY THE PER-PERIOD AUCS
print(f"Periods where AUGMENTED beats BASE:\t{int((g2_dict['quarter_pdf']['d_auc'] > 0).sum())} of {int(g2_dict['quarter_pdf']['d_auc'].notna().sum())}")
g2_dict["quarter_pdf"].round(4)

G2 real:	AUC BASE 0.4233 [0.3648, 0.4829]	AUC AUGMENTED 0.4592 [0.3995, 0.5174]	dAUC 0.0359 [-0.0025, 0.0709] (1-month blocks)	mean per-period dAUC 0.0714
Prediction rows:	2,765	training rows of the first / last fold:	1,408 / 4,112


	null run 1/19: shift 2040 rows, value 0.0203


	null run 2/19: shift 2877 rows, value 0.0342


	null run 3/19: shift 1940 rows, value 0.0043


	null run 4/19: shift 2328 rows, value 0.0210


...

	null run 16/19: shift 2638 rows, value 0.0128


	null run 17/19: shift 1384 rows, value 0.0120


	null run 18/19: shift 2698 rows, value 0.0266


	null run 19/19: shift 3118 rows, value 0.0333
G2 null:	max 0.0563	p = 0.15	PASS:	False	(18 s)
Periods where AUGMENTED beats BASE:	28 of 43


,fold_id,row_count,positive_share,auc_base,auc_augmented,d_auc
0,0,63,0.4762,0.5778,0.4657,-0.1121
1,1,63,0.4603,0.3712,0.4706,0.0994
2,2,64,0.2344,0.1429,0.1986,0.0558
3,3,62,0.9194,0.6491,0.7368,0.0877
4,4,62,0.7258,0.6706,0.6235,-0.0471
5,5,64,0.4062,0.6690,0.8006,0.1316
6,6,63,0.9683,0.3525,0.7705,0.4180
7,7,62,0.6613,0.2950,0.5319,0.2369
8,8,63,0.8095,0.1928,0.4428,0.2500
9,9,65,0.7846,0.5098,0.4748,-0.0350


In [9]:
"""
G3 VOLATILITY FORECAST: does the VIX forecast the next 20 sessions' realized volatility better than the trailing 20-session
realized volatility? (validation decisions; exp05's forward volatility and 6-month circular month blocks)
"""
# RUN THE CHECK
g3_dict = run_g3_dict(primary_pdf, daily_pdf)
assert g3_dict["last_forward_session_date"] <= cutoff_date
g3_passed = g3_dict["passed"]
print(f"G3:\t{g3_dict['row_count']:,} decisions, {g3_dict['month_count']} months\tSpearman VIX {g3_dict['spearman_vix']:.4f}\tSpearman realized {g3_dict['spearman_realized']:.4f}\t"
      f"D {g3_dict['d']:.4f} [{g3_dict['d_ci_low']:.4f}, {g3_dict['d_ci_high']:.4f}]\tlast forward session {g3_dict['last_forward_session_date']}\tPASS:\t{g3_passed}")

G3:	2,765 decisions, 133 months	Spearman VIX 0.6357	Spearman realized 0.5193	D 0.1165 [0.0441, 0.2227]	last forward session 2026-05-13	PASS:	True


In [10]:
"""
G4 PLAUSIBILITY OF UNLEVERED SCALING: an exposure w <= 1 beats buy-and-hold only if what it holds less of earns less
than cash (0%). Mean forward 20-session net return of the rows with vix_level_prev >= the training 80th percentile.
"""
# RUN THE CHECK
g4_dict = run_g4_dict(primary_pdf)
g4_passed = g4_dict["passed"]
print(f"G4:\tthreshold (training 80th percentile of vix_level_prev) {g4_dict['threshold']:.2f}")
print(f"\ttraining:\t{g4_dict['train_row_count']:,} rows, {g4_dict['train_month_count']} months\tmean {g4_dict['train_mean_fwd_return']:.4%} [{g4_dict['train_ci_low']:.4%}, {g4_dict['train_ci_high']:.4%}]")
print(f"\tvalidation:\t{g4_dict['valid_row_count']:,} rows, {g4_dict['valid_month_count']} months\tmean {g4_dict['valid_mean_fwd_return']:.4%} [{g4_dict['valid_ci_low']:.4%}, {g4_dict['valid_ci_high']:.4%}]")
print(f"G4 PASS (both means below 0):\t{g4_passed}")

G4:	threshold (training 80th percentile of vix_level_prev) 22.94
	training:	282 rows, 27 months	mean 1.9141% [0.8948%, 2.9398%]
	validation:	538 rows, 56 months	mean 2.8155% [1.6020%, 3.8968%]
G4 PASS (both means below 0):	False


In [11]:
"""
SECONDARY (reported, never a gate): G1 and G2 with the _intraday features (last 1-minute bar labelled at most decision - 1 minute).
"""
# RUN G1 SECONDARY
start_time = time.time()
g1_secondary_dict = run_g1_dict(secondary_pdf, SECONDARY_COL_STR_LIST)
g1_secondary_null_dict = get_null_dict(secondary_pdf, SECONDARY_COL_STR_LIST, lambda p: run_g1_dict(p, SECONDARY_COL_STR_LIST)["s1"], g1_secondary_dict["s1"], alert_in=False)
g1_secondary_passed = bool(g1_secondary_null_dict["passed"] and g1_secondary_dict["s1"] > 0)
print(f"G1 secondary:\tS1 = {g1_secondary_dict['s1']} of {g1_secondary_dict['test_count']} tested bins {g1_secondary_dict['signal_pair_list']}\tnull values {g1_secondary_null_dict['null_value_list']}\t"
      f"max {g1_secondary_null_dict['null_max']:.0f}\tp = {g1_secondary_null_dict['p_value']:.2f}\twould pass:\t{g1_secondary_passed}")
# RUN G2 SECONDARY
g2_secondary_dict = run_g2_dict(secondary_pdf, fold_pdf, SECONDARY_COL_STR_LIST)
g2_secondary_interval_dict = get_auc_interval_dict(g2_secondary_dict["base_prediction_pdf"], g2_secondary_dict["augmented_prediction_pdf"])
g2_secondary_null_dict = get_null_dict(secondary_pdf, SECONDARY_COL_STR_LIST, lambda p: run_g2_dict(p, fold_pdf, SECONDARY_COL_STR_LIST, base_prediction_pdf_in=g2_secondary_dict["base_prediction_pdf"])["d_auc"],
                                       g2_secondary_dict["d_auc"], alert_in=False)
g2_secondary_passed = bool(g2_secondary_null_dict["passed"])
print(f"G2 secondary:\tAUC BASE {g2_secondary_dict['auc_base']:.4f}\tAUC AUGMENTED {g2_secondary_dict['auc_augmented']:.4f}\tdAUC {g2_secondary_dict['d_auc']:.4f} "
      f"[{g2_secondary_interval_dict['d_auc_ci_low']:.4f}, {g2_secondary_interval_dict['d_auc_ci_high']:.4f}]\tnull max {g2_secondary_null_dict['null_max']:.4f}\tp = {g2_secondary_null_dict['p_value']:.2f}\t"
      f"would pass:\t{g2_secondary_passed}\t({time.time() - start_time:.0f} s)")

G1 secondary:	S1 = 0 of 30 tested bins []	null values [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]	max 0	p = 1.00	would pass:	False


G2 secondary:	AUC BASE 0.4230	AUC AUGMENTED 0.4521	dAUC 0.0291 [-0.0053, 0.0609]	null max 0.0628	p = 0.25	would pass:	False	(23 s)


In [12]:
"""
Gate decisions (pre-registered; primary results only).
"""
# APPLY THE GATE RULES
gate_decision_dict = get_gate_decision_dict(g1_passed, g2_passed, g3_passed, g4_passed)
print(f"PRIMARY:\tG1 {g1_passed}\tG2 {g2_passed}\tG3 {g3_passed}\tG4 {g4_passed}")
print(f"SECONDARY (no gate):\tG1 intraday {g1_secondary_passed}\tG2 intraday {g2_secondary_passed}")
print(f"DECISIONS:\t{gate_decision_dict}")
if not (g1_passed or g2_passed) and (g1_secondary_passed or g2_secondary_passed):
    print("Only a secondary-timing gate would pass: recorded as 'candidate pending review (secondary timing)' and parked; it opens nothing.")
if gate_decision_dict["vix_line_stops"]:
    print("No gate opens: 'no usable VIX information under the pre-registered gates'. The VIX line stops.")

PRIMARY:	G1 False	G2 False	G3 True	G4 False
SECONDARY (no gate):	G1 intraday False	G2 intraday False
DECISIONS:	{'exp09_runs': False, 'exp10_runs': False, 'exp11_runs': False, 'vix_line_stops': True}
No gate opens: 'no usable VIX information under the pre-registered gates'. The VIX line stops.


In [13]:
"""
Descriptive (no gate, no trial): forward 20-session net return by vix_level_prev quintile (training edges) and by
term-structure state, on training and validation; the episodes with vix_term_prev >= 1 (2009-08-12 -> 2026-04-15).
"""
# BUILD THE RETURN TABLE
descriptive_pdf = get_descriptive_return_pdf(primary_pdf)
display(descriptive_pdf.round(4))
# BUILD THE EPISODES (TERM VALUES OF THE DAILY TABLE ROWS; NEXT-20 RETURNS END ON OR BEFORE THE CUTOFF)
episode_pdf = get_term_episode_pdf(daily_pdf, vix_feature_pdf["vix_term_prev"], pd.Timestamp(exp_config.ANALYSIS_START_DATE_STR).date(), pd.to_datetime(fold_pdf["valid_end"].iloc[-1]).date())
print(f"Episodes with vix_term_prev >= {exp_config.TERM_STRESS_THRESHOLD}:\t{len(episode_pdf)}\tsessions {int(episode_pdf['session_count'].sum()) if len(episode_pdf) else 0}")
episode_pdf.round(4)

,grouping,group,window,row_count,month_count,mean_fwd_return,ci_low,ci_high,positive_share
0,vix_level_quintile,"(-inf, 13.874]",train,282,24,0.0044,-0.0040,0.0122,0.6312
1,vix_level_quintile,"(13.874, 16.258]",train,281,41,0.0031,-0.0053,0.0111,0.5801
2,vix_level_quintile,"(16.258, 18.562]",train,282,42,0.0111,0.0000,0.0211,0.7305
3,vix_level_quintile,"(18.562, 22.936]",train,281,39,0.0157,0.0021,0.0272,0.7687
4,vix_level_quintile,"(22.936, inf]",train,282,27,0.0191,0.0089,0.0294,0.7199
5,vix_level_quintile,"(-inf, 13.874]",valid,764,61,0.0049,-0.0018,0.0110,0.6505
6,vix_level_quintile,"(13.874, 16.258]",valid,521,75,0.0047,-0.0064,0.0137,0.6948
7,vix_level_quintile,"(16.258, 18.562]",valid,412,71,0.0080,0.0001,0.0148,0.6845
8,vix_level_quintile,"(18.562, 22.936]",valid,530,74,0.0051,-0.0053,0.0158,0.5981
9,vix_level_quintile,"(22.936, inf]",valid,538,56,0.0282,0.0160,0.0390,0.7398


Episodes with vix_term_prev >= 1.0:	106	sessions 326


,start_date,end_date,session_count,spy_return_episode,spy_return_next_20,next_20_end_date
0,2009-10-29,2009-10-29,1,-0.0008,0.0274,2009-11-27
1,2009-11-02,2009-11-04,3,0.0043,0.0533,2009-12-03
2,2010-01-25,2010-01-26,2,-0.0049,0.0139,2010-02-24
3,2010-02-05,2010-02-09,3,0.0043,0.0722,2010-03-10
4,2010-04-28,2010-04-28,1,0.0006,-0.1020,2010-05-26
...,...,...,...,...,...,...
101,2026-03-09,2026-03-10,2,-0.0021,-0.0017,2026-04-08
102,2026-03-13,2026-03-13,1,0.0000,0.0358,2026-04-13
103,2026-03-24,2026-03-25,2,0.0061,0.0788,2026-04-23
104,2026-03-27,2026-03-31,3,0.0246,0.0942,2026-04-29


In [14]:
# SAVE THE RESULTS
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
write_csv_file_to_path(g1_dict["signal_check_pdf"], f"{output_path_str}g1_signal_check_prev_data.csv", "W", alert_in=False)
write_csv_file_to_path(g1_secondary_dict["signal_check_pdf"], f"{output_path_str}g1_signal_check_intraday_data.csv", "W", alert_in=False)
write_csv_file_to_path(g2_dict["quarter_pdf"], f"{output_path_str}g2_period_auc_prev_data.csv", "W", alert_in=False)
write_csv_file_to_path(g2_dict["base_prediction_pdf"].assign(proba_augmented=g2_dict["augmented_prediction_pdf"]["proba"].to_numpy()), f"{output_path_str}g2_predictions_prev_data.csv", "W", alert_in=False)
write_csv_file_to_path(pd.DataFrame({"run": range(exp_config.NULL_RUN_COUNT), "g1_shift": g1_null_dict["shift_list"], "g1_s1": g1_null_dict["null_value_list"],
                                     "g2_shift": g2_null_dict["shift_list"], "g2_d_auc": g2_null_dict["null_value_list"],
                                     "g1_intraday_s1": g1_secondary_null_dict["null_value_list"], "g2_intraday_d_auc": g2_secondary_null_dict["null_value_list"]}), f"{output_path_str}null_run_data.csv", "W", alert_in=False)
write_csv_file_to_path(descriptive_pdf, f"{output_path_str}descriptive_return_data.csv", "W", alert_in=False)
write_csv_file_to_path(episode_pdf, f"{output_path_str}term_episode_data.csv", "W", alert_in=False)
write_csv_file_to_path(pd.DataFrame([{"g1_passed": g1_passed, "g2_passed": g2_passed, "g3_passed": g3_passed, "g4_passed": g4_passed, "g1_intraday_would_pass": g1_secondary_passed,
                                      "g2_intraday_would_pass": g2_secondary_passed, **gate_decision_dict}]), f"{output_path_str}gate_decision_data.csv", "W", alert_in=False)
print(f"Saved in {output_path_str}")

Saved in C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp08_vix_signal_check/step01_gate_data/

In [15]:
# ASSERT THE TRIAL BUDGET BEFORE LOGGING
assert previous_trial_count + 6 <= exp_config.TRIAL_BUDGET, "❌ The trial budget would be exceeded"
# LOG THE 6 ENTRIES (STAGE "signal_check"; test_window_touched = False ON EVERY ROW)
common_setting_dict = {"run_mode": RUN_MODE_STR, "data_cutoff": exp_config.DATA_CUTOFF_DATE_STR, "analysis_start": exp_config.ANALYSIS_START_DATE_STR}
null_metric_func = lambda d: {"null_max": d["null_max"], "null_ge_count": d["null_ge_count"], "p_value": d["p_value"], "null_value_list": d["null_value_list"]}
trial_list = [
    log_trial_dict(exp_config, "signal_check", {**common_setting_dict, "gate": "G1", "variant": "prev", "features": PRIMARY_COL_STR_LIST},
                   {"s1": g1_dict["s1"], "test_count": g1_dict["test_count"], "signal_pairs": g1_dict["signal_pair_list"], **null_metric_func(g1_null_dict), "passed": g1_passed},
                   False, note_str_in="exp08 G1 univariate signal check (primary _prev) with 19 session-shifted nulls"),
    log_trial_dict(exp_config, "signal_check", {**common_setting_dict, "gate": "G2", "variant": "prev", "features": PRIMARY_COL_STR_LIST},
                   {"auc_base": g2_dict["auc_base"], "auc_augmented": g2_dict["auc_augmented"], "d_auc": g2_dict["d_auc"], "mean_period_d_auc": g2_dict["mean_quarter_d_auc"],
                    **g2_interval_dict, **null_metric_func(g2_null_dict), "passed": g2_passed}, False, note_str_in="exp08 G2 incremental AUC (primary _prev) with 19 session-shifted nulls"),
    log_trial_dict(exp_config, "signal_check", {**common_setting_dict, "gate": "G3", "variant": "prev", "feature": "vix_level_prev"},
                   {k: v for k, v in g3_dict.items()}, False, note_str_in="exp08 G3 volatility forecast (VIX vs trailing realized volatility)"),
    log_trial_dict(exp_config, "signal_check", {**common_setting_dict, "gate": "G4", "variant": "prev", "feature": "vix_level_prev", "top_quantile": exp_config.G4_TOP_QUANTILE},
                   g4_dict, False, note_str_in="exp08 G4 plausibility of unlevered scaling (top VIX quintile forward return < 0 in both windows)"),
    log_trial_dict(exp_config, "signal_check", {**common_setting_dict, "gate": "G1", "variant": "intraday", "features": SECONDARY_COL_STR_LIST},
                   {"s1": g1_secondary_dict["s1"], "test_count": g1_secondary_dict["test_count"], "signal_pairs": g1_secondary_dict["signal_pair_list"], **null_metric_func(g1_secondary_null_dict),
                    "would_pass": g1_secondary_passed}, False, note_str_in="exp08 G1 secondary timing (_intraday; reported, never a gate)"),
    log_trial_dict(exp_config, "signal_check", {**common_setting_dict, "gate": "G2", "variant": "intraday", "features": SECONDARY_COL_STR_LIST},
                   {"auc_base": g2_secondary_dict["auc_base"], "auc_augmented": g2_secondary_dict["auc_augmented"], "d_auc": g2_secondary_dict["d_auc"],
                    **g2_secondary_interval_dict, **null_metric_func(g2_secondary_null_dict), "would_pass": g2_secondary_passed}, False,
                   note_str_in="exp08 G2 secondary timing (_intraday; reported, never a gate)"),
]
print(f"Trials logged:\t{len(trial_list)}\tfirst logged_ts {trial_list[0]['logged_ts']}\ttest_window_touched: {[t['test_window_touched'] for t in trial_list]}")

Trials logged:	6	first logged_ts 2026-10-09T12:55:00	test_window_touched: [False, False, False, False, False, False]


In [16]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,multivariate_check,1,1,0,2026-10-06T18:27:48,2026-10-06T18:27:48
1,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
2,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
3,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
4,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
5,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
6,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
7,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
8,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
9,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
